In [1]:
# Install required libraries (run in Colab if needed)
!pip install transformers datasets gradio torch scikit-learn pandas numpy

import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import gradio as gr

# Step 1: Load and Prepare Data
data = pd.read_csv('/content/text_emotions.csv')
texts = data['content'].values
labels = data['sentiment'].values

# Encode labels
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

# Split dataset
X_train, X_val, y_train, y_val = train_test_split(texts, labels_encoded, test_size=0.2, random_state=42)

# Convert to Hugging Face Dataset format
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
val_df = pd.DataFrame({'text': X_val, 'label': y_val})
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Step 2: Load DistilBERT Tokenizer and Model
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=len(label_encoder.classes_))

# Tokenize function
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

# Tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# Step 3: Define Training Arguments (Optimized for Speed)
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    learning_rate=5e-5,
)

# Step 4: Define Compute Metrics Function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    report = classification_report(labels, preds, target_names=label_encoder.classes_, output_dict=True)
    return {
        'accuracy': (preds == labels).mean(),
        'classification_report': report
    }

# Step 5: Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Step 6: Train the Model
trainer.train()

# Step 7: Evaluate the Model
eval_results = trainer.evaluate()
print("Evaluation Results:")
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")
print("Classification Report:")
for key, value in eval_results['eval_classification_report'].items():
    if isinstance(value, dict):
        print(f"{key}: Precision={value['precision']:.2f}, Recall={value['recall']:.2f}, F1={value['f1-score']:.2f}")

# Step 8: Save the Model
model.save_pretrained('distilbert_emotion_model')
tokenizer.save_pretrained('distilbert_emotion_model')



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Classification Report
1,0.170300,0.172276,0.932000,"{'anger': {'precision': 0.8931034482758621, 'recall': 0.9664179104477612, 'f1-score': 0.9283154121863799, 'support': 536.0}, 'fear': {'precision': 0.9218390804597701, 'recall': 0.8755458515283843, 'f1-score': 0.8980963045912654, 'support': 458.0}, 'joy': {'precision': 0.9681923972071373, 'recall': 0.9320388349514563, 'f1-score': 0.9497716894977168, 'support': 1339.0}, 'love': {'precision': 0.7933673469387755, 'recall': 0.9283582089552239, 'f1-score': 0.8555708390646493, 'support': 335.0}, 'sadness': {'precision': 0.974694589877836, 'recall': 0.9522591645353794, 'f1-score': 0.9633462699439413, 'support': 1173.0}, 'surprise': {'precision': 0.8417721518987342, 'recall': 0.8364779874213837, 'f1-score': 0.8391167192429022, 'support': 159.0}, 'accuracy': 0.932, 'macro avg': {'precision': 0.8988281691096859, 'recall': 0.9151829929732648, 'f1-score': 0.9057028724211426, 'support': 4000.0}, 'weighted avg': {'precision': 0.9350629885724708, 'recall': 0.932, 'f1-score': 0.9326726061906659, 'support': 4000.0}}"
2,0.095100,0.141353,0.931250,"{'anger': {'precision': 0.9130434782608695, 'recall': 0.9402985074626866, 'f1-score': 0.9264705882352942, 'support': 536.0}, 'fear': {'precision': 0.9373433583959899, 'recall': 0.8165938864628821, 'f1-score': 0.8728121353558926, 'support': 458.0}, 'joy': {'precision': 0.9677914110429447, 'recall': 0.9424943988050785, 'f1-score': 0.9549754067347711, 'support': 1339.0}, 'love': {'precision': 0.805699481865285, 'recall': 0.9283582089552239, 'f1-score': 0.8626907073509015, 'support': 335.0}, 'sadness': {'precision': 0.9604377104377104, 'recall': 0.9727195225916454, 'f1-score': 0.9665396018636171, 'support': 1173.0}, 'surprise': {'precision': 0.7777777777777778, 'recall': 0.8364779874213837, 'f1-score': 0.806060606060606, 'support': 159.0}, 'accuracy': 0.93125, 'macro avg': {'precision': 0.8936822029634296, 'recall': 0.9061570852831501, 'f1-score': 0.8982581742668471, 'support': 4000.0}, 'weighted avg': {'precision': 0.9336841723286661, 'recall': 0.93125, 'f1-score': 0.9314910598042964, 'support': 4000.0}}"
3,0.089100,0.163075,0.934000,"{'anger': {'precision': 0.9449715370018975, 'recall': 0.9291044776119403, 'f1-score': 0.9369708372530574, 'support': 536.0}, 'fear': {'precision': 0.956989247311828, 'recall': 0.777292576419214, 'f1-score': 0.8578313253012049, 'support': 458.0}, 'joy': {'precision': 0.9486049926578561, 'recall': 0.9648991784914115, 'f1-score': 0.9566827101073676, 'support': 1339.0}, 'love': {'precision': 0.8892405063291139, 'recall': 0.8388059701492537, 'f1-score': 0.8632872503840245, 'support': 335.0}, 'sadness': {'precision': 0.9561621174524401, 'recall': 0.9855072463768116, 'f1-score': 0.9706129303106633, 'support': 1173.0}, 'surprise': {'precision': 0.7149532710280374, 'recall': 0.9622641509433962, 'f1-score': 0.8203753351206434, 'support': 159.0}, 'accuracy': 0.934, 'macro avg': {'precision': 0.9018202786301955, 'recall': 0.9096455999986711, 'f1-score': 0.9009600647461601, 'support': 4000.0}, 'weighted avg': {'precision': 0.9370348019390317, 'recall': 0.934, 'f1-score': 0.9335677847516487, 'support': 4000.0}}"
4,0.091200,0.169873,0.933500,"{'anger': {'precision': 0.9222423146473779, 'recall': 0.9514925373134329, 'f1-score': 0.9366391184573003, 'support': 536.0}, 'fear': {'precision': 0.9131455399061033, 'recall': 0.8493449781659389, 'f1-score': 0.8800904977375565, 'support': 458.0}, 'joy': {'precision': 0.9594290007513148, 'recall': 0.9536967886482449, 'f1-score': 0.9565543071161049, 'support': 1339.0}, 'love': {'precision': 0.8529411764705882, 'recall': 0.8656716417910447, 'f1-score': 0.8592592592592593, 'support': 335.0}, 'sadness': {'precision': 0.9659863945578231, 'recall': 0.96845694799659, 'f1-score': 0.9672200936568752, 'support': 1173.0}, 'surprise': {'precision': 0.7586206896551724, 'recall': 0.8301886792452831, 'f1-score': 0.7927927927927928, 'support': 159.0}, 'accuracy': 0.9335, 

Trainer is attempting to log a value of "{'anger': {'precision': 0.8931034482758621, 'recall': 0.9664179104477612, 'f1-score': 0.9283154121863799, 'support': 536.0}, 'fear': {'precision': 0.9218390804597701, 'recall': 0.8755458515283843, 'f1-score': 0.8980963045912654, 'support': 458.0}, 'joy': {'precision': 0.9681923972071373, 'recall': 0.9320388349514563, 'f1-score': 0.9497716894977168, 'support': 1339.0}, 'love': {'precision': 0.7933673469387755, 'recall': 0.9283582089552239, 'f1-score': 0.8555708390646493, 'support': 335.0}, 'sadness': {'precision': 0.974694589877836, 'recall': 0.9522591645353794, 'f1-score': 0.9633462699439413, 'support': 1173.0}, 'surprise': {'precision': 0.8417721518987342, 'recall': 0.8364779874213837, 'f1-score': 0.8391167192429022, 'support': 159.0}, 'accuracy': 0.932, 'macro avg': {'precision': 0.8988281691096859, 'recall': 0.9151829929732648, 'f1-score': 0.9057028724211426, 'support': 4000.0}, 'weighted avg': {'precision': 0.9350629885724708, 'recall': 0.93

Trainer is attempting to log a value of "{'anger': {'precision': 0.9339449541284404, 'recall': 0.9496268656716418, 'f1-score': 0.9417206290471786, 'support': 536.0}, 'fear': {'precision': 0.9210526315789473, 'recall': 0.8406113537117904, 'f1-score': 0.8789954337899544, 'support': 458.0}, 'joy': {'precision': 0.9513633014001474, 'recall': 0.964152352501867, 'f1-score': 0.9577151335311572, 'support': 1339.0}, 'love': {'precision': 0.8652694610778443, 'recall': 0.8626865671641791, 'f1-score': 0.8639760837070254, 'support': 335.0}, 'sadness': {'precision': 0.9711129991503823, 'recall': 0.9744245524296675, 'f1-score': 0.9727659574468085, 'support': 1173.0}, 'surprise': {'precision': 0.7692307692307693, 'recall': 0.8176100628930818, 'f1-score': 0.7926829268292683, 'support': 159.0}, 'accuracy': 0.93675, 'macro avg': {'precision': 0.9019956860944219, 'recall': 0.9015186257287047, 'f1-score': 0.9013093607252322, 'support': 4000.0}, 'weighted avg': {'precision': 0.936900142755742, 'recall': 0.9

Evaluation Results:
Accuracy: 0.9367
Classification Report:
anger: Precision=0.93, Recall=0.95, F1=0.94
fear: Precision=0.92, Recall=0.84, F1=0.88
joy: Precision=0.95, Recall=0.96, F1=0.96
love: Precision=0.87, Recall=0.86, F1=0.86
sadness: Precision=0.97, Recall=0.97, F1=0.97
surprise: Precision=0.77, Recall=0.82, F1=0.79
macro avg: Precision=0.90, Recall=0.90, F1=0.90
weighted avg: Precision=0.94, Recall=0.94, F1=0.94


('distilbert_emotion_model/tokenizer_config.json',
 'distilbert_emotion_model/special_tokens_map.json',
 'distilbert_emotion_model/vocab.txt',
 'distilbert_emotion_model/added_tokens.json')

In [4]:
# Step 9: Define Prediction Function
def predict_emotion(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return label_encoder.inverse_transform([predicted_class])[0]

# Step 10: Simple Input Loop for Predictions
def interactive_prediction_loop():
    print("\nEmotion Prediction Loop")
    print("Enter a sentence to predict its emotion. Type 'exit' to quit.")
    while True:
        user_input = input("Enter text: ")
        if user_input.lower() == 'exit':
            print("Exiting prediction loop.")
            break
        if not user_input.strip():
            print("Please enter some text.")
            continue
        emotion = predict_emotion(user_input)
        print(f"Predicted Emotion: {emotion}")

# Run the prediction loop
interactive_prediction_loop()


Emotion Prediction Loop
Enter a sentence to predict its emotion. Type 'exit' to quit.
Enter text: i am very happy today.
Predicted Emotion: joy
Enter text: why  are you acting so strange?
Predicted Emotion: surprise
Enter text: i am very worried
Predicted Emotion: fear
Enter text: you should never do such things again
Predicted Emotion: anger
Enter text: its so pleasant
Predicted Emotion: joy
Enter text: wow! you came to my birthday party
Predicted Emotion: joy
Enter text: exit
Exiting prediction loop.


In [3]:
# Step 8: Save the Model
model.save_pretrained('distilbert_emotion_model')
tokenizer.save_pretrained('distilbert_emotion_model')

('distilbert_emotion_model/tokenizer_config.json',
 'distilbert_emotion_model/special_tokens_map.json',
 'distilbert_emotion_model/vocab.txt',
 'distilbert_emotion_model/added_tokens.json')